In [1]:
import pandas as pd
import numpy as np

In [6]:
pca = pd.read_parquet('../EvaluationResults/Covid-19Vaccine_dataset/comparison_results_PCA.parquet')
tsne = pd.read_parquet('../EvaluationResults/Covid-19Vaccine_dataset/comparison_results_TSNE.parquet')
umap = pd.read_parquet('../EvaluationResults/Covid-19Vaccine_dataset/comparison_results_UMAP.parquet')

In [7]:
umap

,Reference Graph,Compared Graph,Precision,Recall
0,consistency_edges_100_100_100.pkl,umap_edges_100_100_0.0.pkl,0.020270,0.159522
1,consistency_edges_100_100_100.pkl,umap_edges_100_100_0.1.pkl,0.020728,0.161140
2,consistency_edges_100_100_100.pkl,umap_edges_100_100_0.25.pkl,0.019945,0.152778
3,consistency_edges_100_100_100.pkl,umap_edges_100_100_0.5.pkl,0.018409,0.139920
4,consistency_edges_100_100_100.pkl,umap_edges_100_100_0.8.pkl,0.017058,0.129090
...,...,...,...,...
599995,consistency_edges_83_83_83.pkl,umap_edges_83_83_0.1.pkl,0.021050,0.175641
599996,consistency_edges_83_83_83.pkl,umap_edges_83_83_0.25.pkl,0.020178,0.166244
599997,consistency_edges_83_83_83.pkl,umap_edges_83_83_0.5.pkl,0.018885,0.153815
599998,consistency_edges_83_83_83.pkl,umap_edges_83_83_0.8.pkl,0.017160,0.139187


In [5]:
def f1_score(precision: float, recall: float)-> float:
  return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

In [6]:
pca['F1_score'] = pca.apply(lambda row: f1_score(row['Precision'], row['Recall']), axis=1)
tsne['F1_score'] = tsne.apply(lambda row: f1_score(row['Precision'], row['Recall']), axis=1)
umap['F1_score'] = umap.apply(lambda row: f1_score(row['Precision'], row['Recall']), axis=1)

In [7]:
# for i, row in pca.iterrows():
#   print(row['Reference Graph'], row['Compared Graph'], row['f1_score'], '\n')

In [8]:
def get_highest_scores(method: np.ndarray, score: str='F1_score'):
  return method.loc[method.groupby("Reference Graph")[score].idxmax()].set_index('Reference Graph')

In [9]:
pca_highest_f1 = get_highest_scores(pca)
tsne_highest_f1 = get_highest_scores(tsne)
umap_highest_f1 = get_highest_scores(umap)

In [10]:
pca_results = pca_highest_f1[['F1_score']].copy()
tsne_results = tsne_highest_f1[['F1_score']].copy()
umap_results = umap_highest_f1[['F1_score']].copy()

In [11]:
pca_results = pca_results.rename(columns={'F1_score':'pca'})
tsne_results = tsne_results.rename(columns={'F1_score':'tsne'})
umap_results = umap_results.rename(columns={'F1_score':'umap'})

In [12]:
results = pd.concat((pca_results, tsne_results, umap_results), axis=1)

In [13]:
results

,pca,tsne,umap
Reference Graph,,,
consistency_edges_100_100_100.pkl,0.068383,0.157429,0.093523
consistency_edges_100_100_13.pkl,0.070798,0.142574,0.068153
consistency_edges_100_100_23.pkl,0.073434,0.148649,0.071700
consistency_edges_100_100_3.pkl,0.068704,0.136502,0.067482
consistency_edges_100_100_33.pkl,0.074921,0.153161,0.075289
...,...,...,...
consistency_edges_83_83_43.pkl,0.082658,0.167397,0.084385
consistency_edges_83_83_53.pkl,0.082784,0.167651,0.084949
consistency_edges_83_83_63.pkl,0.081763,0.167443,0.088109


In [14]:
from critdd import Diagram

# create a CD diagram from the Pandas DataFrame
diagram = Diagram(
    results.to_numpy(),
    treatment_names = results.columns,
    maximize_outcome = True
)



In [15]:
diagram.average_ranks

array([2.539, 1.   , 2.461])